# Notebook 02: Fixed-Budget Model Scaling (1 vs 2 vs 4 vs 8 vs 16 Experts)
### MOSAIC-Med Research Study: Capacity Budget Allocation under 2.0M Total Parameters

---

### Core Scientific Question & Invariance Envelope
Under a fixed total parameter budget of **~2.0 Million parameters**, how should capacity be distributed?
- **Parameter Invariance Envelope**: $\mathcal{B}_{\text{target}} = 2.00 \times 10^6 \pm 0.06 \times 10^6$ parameters (tolerance $\pm 3.0\%$).
- Channel widths $W$ are discrete integers ($W \in \{16, 23, 32, 46, 65\}$), preventing exact integer convergence to $2,000,000$.
- Configurations:
  - **$K = 1$** ($W=65$): $2,019,945$ params ($+0.997\%$) | 123.9 MFLOPs
  - **$K = 2$** ($W=46$): $2 \times 1,015,225 = 2,030,450$ params ($+1.523\%$) | 126.9 MFLOPs
  - **$K = 4$** ($W=32$): $4 \times 493,893 = 1,975,572$ params ($-1.221\%$) | 126.8 MFLOPs
  - **$K = 8$** ($W=23$): $8 \times 256,869 = 2,054,952$ params ($+2.748\%$) | 136.2 MFLOPs
  - **$K = 16$** ($W=16$): $16 \times 125,605 = 2,009,680$ params ($+0.484\%$) | 139.8 MFLOPs

All configurations are trained across **5 independent random seeds (42–46)** to isolate structural capacity allocation from initialization stochasticity.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("Modules loaded.")


## 1. Scaling Results Table (Seed-Level and Aggregated)
Below are the individual seed results and the aggregated summary statistics (mean ± SD) for all 5 configurations.


In [ ]:
scaling_df = pd.read_csv('results/tables/scaling_results.csv')
scaling_summary = pd.read_csv('results/tables/scaling_summary.csv')

# Add explicit percentage deviation column from 2.0M target
target_budget = 2_000_000
scaling_summary['Budget Deviation (%)'] = ((scaling_summary['Total Parameters'] - target_budget) / target_budget * 100).round(3)

print("Fixed-Budget Scaling Performance Across 5 Seeds:")
scaling_summary[['Configuration', 'Total Parameters', 'Budget Deviation (%)', 'MFLOPs', 'Macro AUROC', 'Macro AUPRC', 'Brier Score', 'Latency (ms)']]


## 2. Discrimination and Calibration Scaling Curves
Let's inspect the scaling curves for AUROC discrimination and Brier score calibration under the fixed 2.0M parameter budget.


In [ ]:
display(Image('results/images/scaling_tradeoff.png'))


## 3. Analysis & Key Insights
1. **Multiplicity Superiority (H1 Supported)**: Partitioning the 2.0M parameter budget into $K=2$ ($0.9085 \pm 0.0007$) and $K=4$ ($0.9084 \pm 0.0009$) compact experts statistically outperforms the monolithic $K=1$ architecture ($0.9059 \pm 0.0010$, DeLong $p = 3.63 \times 10^{-4}$).
2. **Probability Calibration**: Brier scores improve from $0.0900$ ($K=1$) to $0.0881$ ($K=4$), confirming that model splitting acts as an effective variance regularizer.
3. **Over-Fragmentation Boundary ($K=16$)**: At $K=16$ (125K parameters per expert), individual network capacity is insufficient to resolve multi-lead morphological patterns, degrading AUROC to $0.9048 \pm 0.0005$ and tripling latency to $6.78$ ms.
4. **Capacity vs Compute**: Parameter counts are matched within $\pm 3.0\%$, but sequential inference FLOPs scale slightly from 123.9 to 139.8 MFLOPs due to per-network stem overheads.
